# 04 · Practice offline with the sandbox site

Start the bundled practice website in-process and try logins with CSRF tokens, authenticated JSON APIs, sitemaps and retries — no internet needed.

*Jalankan situs latihan bawaan dan coba login dengan CSRF, API JSON berautentikasi, sitemap, dan retry — tanpa internet.*

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
// Packages / Paket
#r "nuget: Gravicode.ScrapyNet, 0.1.0"
#r "nuget: Gravicode.ScrapyNet.Sandbox, 0.1.0"

// Before the packages are on nuget.org, point at a local build instead:
//   dotnet pack ScrapyNet.slnx -c Release -o artifacts
// and add this line above the #r lines:  #i "nuget: ../artifacts"

In [ ]:
using ScrapyNet;
using ScrapyNet.Sandbox;

var site = await SandboxSite.StartAsync();
display(site.BaseUrl);

var quiet = new Settings();
quiet.Set(SettingKeys.LogLevel, "WARNING");
quiet.Set(SettingKeys.StatsDump, false);

## Log in with a form

`FormRequest.FromResponse` keeps hidden fields such as the CSRF token; the cookie middleware keeps the session.

*`FormRequest.FromResponse` mempertahankan field tersembunyi seperti token CSRF; middleware cookie menjaga sesi.*

In [ ]:
public sealed class LoginSpider(string url) : Spider
{
    public override IReadOnlyList<string> StartUrls => [url];

    public override async IAsyncEnumerable<object> Parse(Response response)
    {
        yield return FormRequest.FromResponse(response,
            [new("username", SandboxSite.Username), new("password", SandboxSite.Password)],
            callback: Account);
        await Task.CompletedTask;
    }

    private IEnumerable<object> Account(Response r)
    {
        yield return new { Welcome = r.Css("h1.welcome::text").Get(), Orders = r.Css("li.order").Count };
    }
}

In [ ]:
var (_, account) = await Scrapy.CrawlAndCollectAsync(new LoginSpider(site.Url("/login")), quiet);
display(ItemAdapter.For(account[0]));

## Retries on a flaky endpoint

`/flaky/{key}?fail=2` answers 503 twice before succeeding; the retry middleware handles it.

*Endpoint ini gagal dua kali sebelum berhasil; middleware retry menanganinya.*

In [ ]:
var flaky = await ScrapyNet.Cmdline.Fetcher.FetchAsync(site.Url($"/flaky/{Guid.NewGuid():N}?fail=2"));
display($"{flaky.Status} after {flaky.Request.Meta.GetValueOrDefault(MetaKeys.RetryTimes)} retries");
await site.DisposeAsync();